In [0]:

# ============================================================
# BRONZE TO SILVER NOTEBOOK
# Catalog: databrickscaresync
# Supported load types: FULL, APPEND, MERGE
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ------------------------------------------------------------
# STEP 0: NOTEBOOK PARAMETERS
# ------------------------------------------------------------

# Widget names must match the ADF base parameters: tablename, landingtimestamp, merge_keys, load_type
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp").strip()
table_name = dbutils.widgets.get("table_name").strip()
merge_keys = dbutils.widgets.get("merge_keys").strip()
load_type = dbutils.widgets.get("load_type").strip()

CATALOG = "databrickscaresync"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

print("========== NOTEBOOK PARAMETERS ==========")
print("Catalog:", CATALOG)
print("Table:", table_name)
print("Landing timestamp:", landing_timestamp)
print("Load type:", load_type)
print("Merge keys:", merge_keys)

# ------------------------------------------------------------
# STEP 1: PARAMETER VALIDATION
# ------------------------------------------------------------

if not table_name:
    raise ValueError("tablename parameter cannot be empty.")

if not landing_timestamp:
    raise ValueError("landingtimestamp parameter cannot be empty.")

normalized_load_type = (load_type or "FULL").upper()

if normalized_load_type not in ["FULL", "APPEND", "MERGE"]:
    raise ValueError(
        "load_type must be FULL, APPEND, or MERGE."
    )

# Restrict table name to a simple SQL identifier.
if not table_name.replace("_", "").isalnum():
    raise ValueError(f"Invalid table name: {table_name}")

source_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
target_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

merge_key_list = [
    key.strip() for key in merge_keys.split(",") if key.strip()
]

print("\n[SOURCE TABLE]", source_table)
print("[TARGET TABLE]", target_table)

# ------------------------------------------------------------
# STEP 2: READ BRONZE DATA
# ------------------------------------------------------------

print("\n[STEP 2] Reading Bronze data")

if not spark.catalog.tableExists(source_table):
    raise ValueError(f"Bronze table does not exist: {source_table}")

bronze_df = (
    spark.read.table(source_table)
    .filter(F.col("landing_timestamp") == landing_timestamp)
)

source_count = bronze_df.count()

print("[BRONZE ROW COUNT]", source_count)
print("[BRONZE COLUMNS]", bronze_df.columns)

if source_count == 0:
    raise ValueError(
        f"No Bronze records found for landing_timestamp: "
        f"{landing_timestamp}"
    )

# ------------------------------------------------------------
# STEP 3: ADD SILVER AUDIT COLUMNS
# ------------------------------------------------------------


# ------------------------------------------------------------
# STEP 3: ADD SILVER AUDIT COLUMNS
# ------------------------------------------------------------

from pyspark.sql import functions as F

print("\n[STEP 3] Preparing Silver DataFrame")

updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("\n[UPDATED DATAFRAME SCHEMA]")
updated_df.printSchema()

print("\n[UPDATED DATAFRAME COLUMNS]")
print(updated_df.columns)

# Safety validation
required_audit_columns = [
    "insert_timestamp",
    "update_timestamp"
]

missing_audit_columns = [
    column for column in required_audit_columns
    if column not in updated_df.columns
]

if missing_audit_columns:
    raise ValueError(
        f"Audit columns were not added: {missing_audit_columns}"
    )

print("\n[STEP 3 COMPLETE] Both audit columns are present.")

# ------------------------------------------------------------
# STEP 4: CHECK TARGET TABLE
# ------------------------------------------------------------

print("\n[STEP 4] Checking Silver target")

target_exists = spark.catalog.tableExists(target_table)

print("[TARGET EXISTS]", target_exists)

if target_exists:
    target_df = spark.read.table(target_table)

    source_columns = set(updated_df.columns)
    target_columns = set(target_df.columns)

    print("[SOURCE COLUMNS]", sorted(source_columns))
    print("[TARGET COLUMNS]", sorted(target_columns))

    missing_in_source = sorted(target_columns - source_columns)
    additional_in_source = sorted(source_columns - target_columns)

    print("[MISSING IN SOURCE]", missing_in_source)
    print("[ADDITIONAL IN SOURCE]", additional_in_source)

# ------------------------------------------------------------
# STEP 5: VALIDATE MERGE KEYS
# ------------------------------------------------------------

if normalized_load_type == "MERGE":

    print("\n[STEP 5] Validating MERGE keys")

    if not merge_key_list:
        raise ValueError(
            "merge_keys must be provided for MERGE."
        )

    missing_keys = [
        key for key in merge_key_list
        if key not in updated_df.columns
    ]

    if missing_keys:
        raise ValueError(
            f"MERGE keys missing from source: {missing_keys}"
        )

    if target_exists:
        missing_target_keys = [
            key for key in merge_key_list
            if key not in target_df.columns
        ]

        if missing_target_keys:
            raise ValueError(
                f"MERGE keys missing from target: "
                f"{missing_target_keys}"
            )

    print("[MERGE KEYS VALIDATED]", merge_key_list)

# ------------------------------------------------------------
# STEP 6: FULL LOAD
# ------------------------------------------------------------

if normalized_load_type == "FULL":

    print("\n[STEP 6] FULL LOAD")

    if target_exists:
        # Do not overwrite a table with an incompatible schema.
        source_schema = {
            field.name: field.dataType
            for field in updated_df.schema.fields
        }

        target_schema = {
            field.name: field.dataType
            for field in target_df.schema.fields
        }

        if source_schema != target_schema:
            raise ValueError(
                "FULL LOAD STOPPED: Source and target schemas differ. "
                "No target table was modified. "
                f"Source schema: {source_schema}. "
                f"Target schema: {target_schema}."
            )

    # This writes the complete current batch.
    # Existing target replacement is allowed only after exact schema validation.
    updated_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    print("[FULL LOAD COMPLETED]")

# ------------------------------------------------------------
# STEP 7: APPEND LOAD
# ------------------------------------------------------------

elif normalized_load_type == "APPEND":

    print("\n[STEP 7] APPEND LOAD")

    if target_exists:
        source_schema = {
            field.name: field.dataType
            for field in updated_df.schema.fields
        }

        target_schema = {
            field.name: field.dataType
            for field in target_df.schema.fields
        }

        if source_schema != target_schema:
            raise ValueError(
                "APPEND STOPPED: Source and target schemas differ. "
                "No records were appended."
            )

    updated_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

    print("[APPEND LOAD COMPLETED]")

# ------------------------------------------------------------
# STEP 8: MERGE LOAD
# ------------------------------------------------------------

elif normalized_load_type == "MERGE":

    print("\n[STEP 8] MERGE LOAD")

    if not target_exists:

        print("[INITIAL MERGE] Target does not exist.")

        updated_df.write \
            .format("delta") \
            .mode("errorifexists") \
            .saveAsTable(target_table)

        print("[INITIAL MERGE TABLE CREATED]")

    else:

        source_schema = {
            field.name: field.dataType
            for field in updated_df.schema.fields
        }

        target_schema = {
            field.name: field.dataType
            for field in target_df.schema.fields
        }

        if source_schema != target_schema:
            raise ValueError(
                "MERGE STOPPED: Source and target schemas differ. "
                "No MERGE was executed."
            )

        # Check for duplicate source keys.
        duplicate_keys = (
            updated_df.groupBy(*merge_key_list)
            .count()
            .filter(F.col("count") > 1)
        )

        if duplicate_keys.limit(1).count() > 0:
            raise ValueError(
                "MERGE STOPPED: Duplicate merge keys found in source."
            )

        # Preserve insert_timestamp for matched records.
        source_view = "silver_merge_source"

        updated_df.createOrReplaceTempView(source_view)

        merge_condition = " AND ".join(
            [
                f"target.`{key}` = source.`{key}`"
                for key in merge_key_list
            ]
        )

        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )

        insert_columns = ", ".join(
            [f"`{column}`" for column in updated_df.columns]
        )

        insert_values = ", ".join(
            [f"source.`{column}`" for column in updated_df.columns]
        )

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}

            WHEN MATCHED THEN
                UPDATE SET {update_assignments}

            WHEN NOT MATCHED THEN
                INSERT ({insert_columns})
                VALUES ({insert_values})
        """

        print("[MERGE SQL]")
        print(merge_sql)

        spark.sql(merge_sql)

        print("[MERGE COMPLETED]")

# ------------------------------------------------------------
# STEP 9: VALIDATE TARGET AFTER WRITE
# ------------------------------------------------------------

print("\n[STEP 9] Validating Silver target")

final_df = spark.read.table(target_table)

final_count = final_df.count()

print("[SOURCE BATCH COUNT]", source_count)
print("[SILVER TABLE COUNT]", final_count)
print("[FINAL SILVER COLUMNS]", final_df.columns)

print("\n========== BRONZE TO SILVER COMPLETED ==========")

dbutils.notebook.exit(str(source_count))


In [0]:
table_name = dbutils.widgets.get("tablename").strip()

if not table_name:
    raise ValueError("tablename parameter cannot be empty.")